# 01 — Agente Simples: Padrão ReAct

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 05_Agentes  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- **Padrão ReAct** — o ciclo Reason → Act → Observe que define um agente
- **tool_runner.py** — como o shared/ já implementa function calling com fallback para DeepSeek
- **Loop de agente** — controle do número de iterações, detecção de resposta final
- **Ferramentas básicas** — calculadora, data/hora, consulta ao curso

---

### O que é um agente?

Um LLM sozinho responde com o que sabe. Um **agente** decide *quando* e *como* usar ferramentas
para buscar informação ou executar ações — e itera até ter uma resposta completa.

```
Pergunta do usuário
        │
        ▼
   ┌─── LLM raciocina ───────────────────────────────┐
   │    (Reason)                                      │
   │         │                                        │
   │         ▼                                        │
   │    Precisa de ferramenta? ──Não──→ Resposta final│
   │         │ Sim                                    │
   │         ▼                                        │
   │    Chama ferramenta  (Act)                       │
   │         │                                        │
   │         ▼                                        │
   │    Recebe resultado  (Observe)                   │
   └─────────────────────────────────────────────────┘
```

Esse ciclo é o **padrão ReAct** (Reasoning + Acting), base da maioria dos agentes modernos.

## Setup

In [20]:
import sys, os, json, time, math
from datetime import datetime
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

# tool_runner expõe executar_com_tools e ToolRunner
from shared.tool_runner import executar_com_tools, ToolRunner

import os
LLM_MODEL    = os.getenv('LLM_MODEL', 'deepseek-chat')
PROJETO_BASE = os.path.abspath('../..')

print(f'LLM     : {LLM_MODEL}')
print(f'Projeto : {PROJETO_BASE}')
print(f'tool_runner: executar_com_tools + ToolRunner carregados')


LLM     : deepseek-chat
Projeto : C:\Users\Jorge Maques\Documents\Especialista_em_AI
tool_runner: executar_com_tools + ToolRunner carregados


---
## 1. Definição das Ferramentas

Ferramentas são funções Python descritas no formato JSON Schema que o LLM usa para decidir
quando e como chamá-las. Cada ferramenta tem:
- **Schema** — descrição, parâmetros e tipos (passado ao LLM)
- **Implementação** — a função Python que executa de fato
- **Registro** — dicionário que mapeia nome → função

In [21]:
# ── Implementações das ferramentas ────────────────────────────────────────

def calcular(expressao: str) -> str:
    """
    Avalia uma expressão matemática de forma segura.
    Permite: +, -, *, /, **, sqrt, log, sin, cos, pi, e
    """
    permitidos = {
        'sqrt': math.sqrt, 'log': math.log, 'log10': math.log10,
        'sin': math.sin,   'cos': math.cos, 'tan': math.tan,
        'pi': math.pi,     'e': math.e,     'abs': abs,
        'round': round,    'pow': pow,
    }
    try:
        resultado = eval(expressao, {'__builtins__': {}}, permitidos)
        return f'{resultado}'
    except Exception as ex:
        return f'Erro ao calcular "{expressao}": {ex}'


def data_hora_atual(formato: str = '%d/%m/%Y %H:%M:%S') -> str:
    """Retorna a data e hora atual formatada."""
    return datetime.now().strftime(formato)


def consultar_curso(pergunta: str, modulo: str = '') -> str:
    """
    Consulta informações do curso via busca nos AGENT_CONTEXT.md.
    Retorna trechos relevantes encontrados nos arquivos.
    """
    # Tokeniza a pergunta — ignora stopwords curtas
    palavras = [p for p in pergunta.lower().split() if len(p) > 3]
    ignorar  = {'.git', 'venv', '.venv', '__pycache__', '.ipynb_checkpoints'}
    resultados = []

    for raiz, dirs, arquivos in os.walk(PROJETO_BASE):
        dirs[:] = [d for d in dirs if d not in ignorar]
        if 'AGENT_CONTEXT.md' not in arquivos:
            continue
        partes = raiz.replace('\\\\', '/').replace('\\', '/').split('/')
        mod    = next((p for p in partes if p.startswith('EAI_')), '')
        if modulo and not mod.startswith(modulo):
            continue
        caminho = os.path.join(raiz, 'AGENT_CONTEXT.md')
        with open(caminho, encoding='utf-8') as f:
            linhas = f.readlines()
        for linha in linhas:
            linha_lower = linha.lower()
            # Conta quantas palavras da pergunta aparecem na linha
            hits = sum(1 for p in palavras if p in linha_lower)
            if hits >= min(2, len(palavras)):  # pelo menos 2 palavras (ou 1 se pergunta curta)
                resultados.append((hits, f'[{mod}] {linha.strip()}'))

    if not resultados:
        # Fallback: busca por qualquer palavra
        for raiz, dirs, arquivos in os.walk(PROJETO_BASE):
            dirs[:] = [d for d in dirs if d not in ignorar]
            if 'AGENT_CONTEXT.md' not in arquivos:
                continue
            partes = raiz.replace('\\\\', '/').replace('\\', '/').split('/')
            mod    = next((p for p in partes if p.startswith('EAI_')), '')
            if modulo and not mod.startswith(modulo):
                continue
            caminho = os.path.join(raiz, 'AGENT_CONTEXT.md')
            with open(caminho, encoding='utf-8') as f:
                linhas = f.readlines()
            for linha in linhas:
                if any(p in linha.lower() for p in palavras):
                    resultados.append((1, f'[{mod}] {linha.strip()}'))

    if not resultados:
        return f'Nenhuma informação encontrada para: {pergunta}'

    # Ordena por relevância (mais hits primeiro) e retorna top 8
    resultados.sort(key=lambda x: x[0], reverse=True)
    return '\n'.join(r for _, r in resultados[:8])


# ── Registro: nome → função ───────────────────────────────────────────────
FERRAMENTAS_FN = {
    'calcular'       : calcular,
    'data_hora_atual': data_hora_atual,
    'consultar_curso': consultar_curso,
}

print('Ferramentas registradas:', list(FERRAMENTAS_FN.keys()))


Ferramentas registradas: ['calcular', 'data_hora_atual', 'consultar_curso']


In [22]:
# ── Schemas JSON para o LLM ───────────────────────────────────────────────

FERRAMENTAS_SCHEMA = [
    {
        'type': 'function',
        'function': {
            'name': 'calcular',
            'description': 'Avalia expressões matemáticas. Use para cálculos numéricos, '
                           'conversões e operações com sqrt, log, sin, cos, pi.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'expressao': {
                        'type': 'string',
                        'description': 'Expressão matemática válida em Python. Ex: "2**10", "sqrt(144)", "pi * 5**2"'
                    }
                },
                'required': ['expressao']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'data_hora_atual',
            'description': 'Retorna a data e hora atual do sistema.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'formato': {
                        'type': 'string',
                        'description': 'Formato strftime. Padrão: "%d/%m/%Y %H:%M:%S"'
                    }
                },
                'required': []
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'consultar_curso',
            'description': 'Busca informações técnicas sobre os módulos do curso '
                           'Especialista em IA (EAI_01 a EAI_08).',
            'parameters': {
                'type': 'object',
                'properties': {
                    'pergunta': {
                        'type': 'string',
                        'description': 'Pergunta ou termos de busca sobre o curso'
                    },
                    'modulo': {
                        'type': 'string',
                        'description': 'Filtro opcional de módulo, ex: "EAI_01", "EAI_07"'
                    }
                },
                'required': ['pergunta']
            }
        }
    },
]

print(f'{len(FERRAMENTAS_SCHEMA)} schemas definidos.')

3 schemas definidos.


---
## 2. Loop ReAct

O loop ReAct é o coração do agente. A cada iteração:

1. **Reason** — LLM recebe o histórico e decide o próximo passo
2. **Act** — se decidiu usar ferramenta, executa via `tool_runner`
3. **Observe** — resultado da ferramenta é adicionado ao histórico
4. Repete até o LLM emitir resposta final (sem tool call)

O `tool_runner.py` do `shared/` já trata o caso DeepSeek onde a resposta
de tool call não segue o formato padrão OpenAI.

In [23]:
SYSTEM_AGENTE = """\
Você é um assistente técnico inteligente do curso Especialista em IA.
Você tem acesso a ferramentas e deve usá-las quando necessário para dar respostas precisas.

Diretrizes:
- Use calcular para qualquer operação matemática — não calcule de cabeça
- Use data_hora_atual quando a pergunta envolver datas ou tempo
- Use consultar_curso para perguntas sobre conteúdo dos módulos EAI_01 a EAI_08
- Você pode encadear múltiplas ferramentas na mesma resposta
- Quando tiver informação suficiente, responda diretamente sem chamar mais ferramentas
"""

# ── Monta o agente com ToolRunner ────────────────────────────────────────
# ToolRunner.registrar(schema, funcao) — registra cada ferramenta
# ToolRunner.perguntar(texto)          — dispara o loop ReAct completo

agente = ToolRunner(system=SYSTEM_AGENTE, verbose=True)

agente.registrar(FERRAMENTAS_SCHEMA[0], calcular)
agente.registrar(FERRAMENTAS_SCHEMA[1], data_hora_atual)
agente.registrar(FERRAMENTAS_SCHEMA[2], consultar_curso)

print(agente)


ToolRunner(tools=['calcular', 'data_hora_atual', 'consultar_curso'])


---
## 3. Testes do Agente

Testamos três cenários que exigem ferramentas diferentes e encadeamento.

In [24]:
print('='*60)
print('TESTE 1 — Cálculo')
print('='*60)
pergunta = 'Qual é a raiz quadrada de 1764 elevada ao quadrado dividido por 7?'
print(f'👤 {pergunta}\n')
resp = agente.perguntar(pergunta)
print(f'\n🤖 {resp}')


TESTE 1 — Cálculo
👤 Qual é a raiz quadrada de 1764 elevada ao quadrado dividido por 7?

[iter 1] 1 tool call(s) — OpenAI
  -> calcular({'expressao': 'sqrt(1764)'})
     = 42.0
[iter 2] 1 tool call(s) — OpenAI
  -> calcular({'expressao': '42**2 / 7'})
     = 252.0
[iter 3] 1 tool call(s) — OpenAI
  -> calcular({'expressao': '1764 / 7'})
     = 252.0
[iter 4] Resposta final

🤖 Perfeito! A resposta é **252**.

**Explicação:**
1. Primeiro calculamos a raiz quadrada de 1764: √1764 = 42
2. Depois elevamos ao quadrado: 42² = 1764
3. Finalmente dividimos por 7: 1764 ÷ 7 = 252

Note que há uma simplificação interessante: (√1764)² ÷ 7 = 1764 ÷ 7 = 252, pois (√a)² = a.


In [25]:
print('='*60)
print('TESTE 2 — Consulta ao curso')
print('='*60)
pergunta = 'Que técnicas de RAG avançado foram implementadas no EAI_07?'
print(f'👤 {pergunta}\n')
resp = agente.perguntar(pergunta)
print(f'\n🤖 {resp}')


TESTE 2 — Consulta ao curso
👤 Que técnicas de RAG avançado foram implementadas no EAI_07?

[iter 1] 1 tool call(s) — OpenAI
  -> consultar_curso({'pergunta': 'técnicas de RAG avançado implementadas', 'modulo': 'EAI_07'})
     = [EAI_07_AI_Generative] **Técnicas implementadas**:
[iter 2] 1 tool call(s) — OpenAI
  -> consultar_curso({'pergunta': 'RAG avançado técnicas implementadas', 'modulo': 'EAI_07'})
     = [EAI_07_AI_Generative] **Técnicas implementadas**:
[iter 3] 1 tool call(s) — OpenAI
  -> consultar_curso({'pergunta': 'RAG retrieval augmented generation avançado', 'modulo': 'EAI_07'})
     = [EAI_07_AI_Generative] function calling RAG retrieval augmented generation embeddings FAISS BM25 busca híbrida
[EAI_07_AI_Generative] Implementação progressiva de RAG (Retrieval Augmented Generation) em 5 notebooks,
[EAI_07_AI_Generative] RAG retrieval augmented generation busca semântica FAISS embeddings sentence-transformers
[iter 4] 1 tool call(s) — OpenAI
  -> consultar_curso({'pergunta'

In [26]:
# Teste 3: encadeamento de ferramentas
print('='*60)
print('TESTE 3 — Encadeamento')
print('='*60)
pergunta = (
    'Que horas são agora? '
    'E por que o MobileNetV2 foi escolhido no EAI_03 em vez do ResNet? '
    'Calcule também a diferença de tamanho em MB entre os dois modelos (14 MB vs 90 MB).'
)
print(f'👤 {pergunta}\n')
resp = agente.perguntar(pergunta)
print(f'\n🤖 {resp}')


TESTE 3 — Encadeamento
👤 Que horas são agora? E por que o MobileNetV2 foi escolhido no EAI_03 em vez do ResNet? Calcule também a diferença de tamanho em MB entre os dois modelos (14 MB vs 90 MB).

[iter 1] 2 tool call(s) — OpenAI
  -> data_hora_atual({})
     = 31/03/2026 06:53:03
  -> consultar_curso({'pergunta': 'MobileNetV2 escolhido em vez do ResNet EAI_03', 'modulo': 'EAI_03'})
     = [EAI_03_Deep_Learning] - [ ] Ensemble: Combinar MobileNetV2 + EfficientNet + ResNet
[EAI_03_Deep_Learning] **Q: Por que MobileNetV2 em vez de ResNet/VGG?**
[EAI_03_Deep_Learning] A: MobileNetV2 é leve (~14 MB vs ~90 MB ResNet50), rápida, e suficiente para deployment. Ideal para Flask em CPU.
[iter 2] 1 tool call(s) — OpenAI
  -> calcular({'expressao': '90 - 14'})
     = 76
[iter 3] Resposta final

🤖 Aqui estão as respostas para suas perguntas:

## 1. Horário atual
São **06:53:03** do dia **31/03/2026**.

## 2. Por que MobileNetV2 foi escolhido em vez do ResNet no EAI_03
De acordo com o conteúdo do cu

---
## 4. Inspecionando o Loop

Versão usando `executar_com_tools` diretamente — mostra o log interno do `tool_runner`
com número de iterações e ferramentas chamadas em cada etapa.

> **Limitação da `consultar_curso`:** usa busca por keywords simples.
> Funciona bem para termos técnicos exatos (nomes de modelos, algoritmos, módulos).
> Para perguntas semânticas como *"qual foi a acurácia?"*, o índice FAISS do `03_RAG`
> é mais adequado — no `02_ferramentas_customizadas.ipynb` integramos o RAG como ferramenta.


In [27]:
# ── Seção 4: inspecionando o loop com executar_com_tools ─────────────────
#
# Nota: consultar_curso usa busca por keywords — funciona bem para perguntas
# com termos técnicos exatos (nomes de modelos, algoritmos, módulos).
# Para busca semântica completa, use o índice FAISS do 03_RAG.

perguntas = [
    'Quanto é 15 elevado a 3?',
    'Por que o MobileNetV2 foi escolhido no EAI_03 em vez do ResNet?',
]

for p in perguntas:
    print('='*55)
    print(f'Pergunta: {p}')
    print('-'*55)
    resp = executar_com_tools(
        pergunta  = p,
        tools     = FERRAMENTAS_SCHEMA,
        funcoes   = FERRAMENTAS_FN,
        system    = SYSTEM_AGENTE,
        verbose   = True,
    )
    print(f'\nResposta: {resp}')
    print()


Pergunta: Quanto é 15 elevado a 3?
-------------------------------------------------------
[iter 1] 1 tool call(s) — OpenAI
  -> calcular({'expressao': '15**3'})
     = 3375
[iter 2] Resposta final

Resposta: 15 elevado a 3 é igual a **3375**.

Pergunta: Por que o MobileNetV2 foi escolhido no EAI_03 em vez do ResNet?
-------------------------------------------------------
[iter 1] 1 tool call(s) — OpenAI
  -> consultar_curso({'pergunta': 'MobileNetV2 ResNet comparação escolha EAI_03', 'modulo': 'EAI_03'})
     = [EAI_03_Deep_Learning] - [ ] Ensemble: Combinar MobileNetV2 + EfficientNet + ResNet
[EAI_03_Deep_Learning] **Q: Por que MobileNetV2 em vez de ResNet/VGG?**
[EAI_03_Deep_Learning] A: MobileNetV2 é leve (~14 MB vs ~90 MB ResNet50), rápida, e suficiente para deployment. Ideal para Flask em CPU.
[iter 2] Resposta final

Resposta: Com base na consulta ao conteúdo do curso EAI_03, a escolha do MobileNetV2 em vez do ResNet foi baseada em **três fatores principais**:

1. **Tamanho do m

---
## Resumo

| Conceito | Implementação |
|---|---|
| **Reason** | LLM recebe histórico + schemas e decide: responder ou chamar ferramenta |
| **Act** | `ToolRunner.perguntar()` despacha para a função Python registrada |
| **Observe** | Resultado adicionado ao histórico como `role='tool'` |
| **Loop** | `executar_com_tools()` itera até sem tool calls ou `max_iteracoes=8` |
| **DSML fallback** | `tool_runner.py` trata respostas não-padrão do DeepSeek automaticamente |

### Fluxo de uso

```python
# 1. Cria o agente
agente = ToolRunner(system=SYSTEM_AGENTE, verbose=True)

# 2. Registra ferramentas (schema + função Python)
agente.registrar(FERRAMENTAS_SCHEMA[0], calcular)

# 3. Faz perguntas — loop ReAct completo acontece internamente
resposta = agente.perguntar('Qual é a raiz de 144?')
```

### Por que limitar iterações?

Sem `max_iteracoes`, um agente pode entrar em loop se uma ferramenta retorna
resultado ambíguo e o LLM continua chamando mais ferramentas sem convergir.
8 iterações cobre a grande maioria dos casos práticos.
